In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout

In [ ]:
# ── Step 1: Load Dataset ─────────────────────────────────────────────────────
# Google stock prices — we will predict the 'Open' price using past 5 days
data = pd.read_csv("Datasets/google.csv")
data.head()

In [ ]:
# ── Step 2: Select Column, Split, and Scale ───────────────────────────────────
# We only use the 'Open' price column for this univariate time-series prediction
training_data = data[["Open"]]

# Split by index (NOT random) — time order must be preserved
split_index   = int(len(training_data) * 0.8)
training_data = training_data[:split_index]

# Scale values to [0, 1] — LSTMs train much better on normalised data
# fit_transform on train only; test will use sc.transform (not sc.fit_transform)
sc = MinMaxScaler(feature_range=(0, 1))
scaled_training_data = sc.fit_transform(training_data)

In [ ]:
# ── Step 3: Create Sliding Window (Timesteps) ────────────────────────────────
# For each day i, the input is the past 5 days' prices → output is day i's price
# e.g. x_train[0] = days 0-4, y_train[0] = day 5
x_train = []
y_train = []

for i in range(5, len(scaled_training_data)):
    x_train.append(scaled_training_data[i-5:i, 0])   # past 5 days
    y_train.append(scaled_training_data[i, 0])         # current day

# Convert to numpy arrays — required by TensorFlow
x_train = np.array(x_train)
y_train = np.array(y_train)

In [ ]:
# ── Step 4: Reshape for LSTM ─────────────────────────────────────────────────
# LSTM expects shape: (samples, timesteps, features)
# Our data: (num_windows, 5 days, 1 feature)
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))
print("x_train shape:", x_train.shape)  # should be (N, 5, 1)

In [ ]:
# ── Step 5: Build the LSTM Model ─────────────────────────────────────────────
# LSTM: Long Short-Term Memory — remembers patterns across a sequence of steps
# return_sequences=True: pass the full sequence to the next LSTM (not just last step)
# return_sequences=False on the last LSTM: output only the final step
# Dropout(0.2): randomly drops 20% of neurons to reduce overfitting
# Dense(1): single output — the predicted next price (still scaled)
model = Sequential()

model.add(LSTM(units=50, return_sequences=True, input_shape=(x_train.shape[1], 1)))
model.add(Dropout(0.2))

model.add(LSTM(units=50, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(units=50, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(units=50, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(units=50))   # last LSTM: no return_sequences
model.add(Dropout(0.2))

model.add(Dense(units=1))   # output: predicted price (scaled)

# mse loss: standard for regression; no extra metrics needed here
model.compile(optimizer='adam', loss='mean_squared_error')

In [ ]:
# ── Step 6: Train ────────────────────────────────────────────────────────────
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=4,
    verbose=1
)

In [ ]:
# ── Step 7: Prepare Test Data and Predict ────────────────────────────────────
# We need the 5 days BEFORE the test period to form the first test window
test_data        = data[["Open"]][split_index:]
real_stock_price = test_data.values

dataset_total = data[["Open"]]

# Take last 5 training days + all test days as input context
inputs = dataset_total[len(dataset_total) - len(test_data) - 5:].values
inputs = sc.transform(inputs)   # scale using the SAME scaler fitted on training data

# Build sliding windows for test (same logic as training)
X_test = []
for i in range(5, len(inputs)):
    X_test.append(inputs[i-5:i, 0])

X_test = np.array(X_test)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

# Predict scaled prices, then inverse transform back to original USD scale
predicted_stock_price = model.predict(X_test)
predicted_stock_price = sc.inverse_transform(predicted_stock_price)

In [ ]:
# ── Step 8: Visualise Results ─────────────────────────────────────────────────
# Red = actual prices, Blue = model predictions
# A good model: the blue line should closely follow the red line
plt.plot(real_stock_price,        color='red',  label='Real Stock Price')
plt.plot(predicted_stock_price,   color='blue', label='Predicted Stock Price')
plt.title('Google Stock Price Prediction')
plt.xlabel('Time')
plt.ylabel('Stock Price (USD)')
plt.legend()
plt.show()